# 2. Arm one: the models

Reproduces *RQ1 on arm one*, *Against chance, baselines and the ceiling*, *Feature interpretation*
and *RQ2 screening utility*. The estimator settings are those of the frozen analysis, and nothing is tuned or
selected here: every learner is run under the same nested, group-disjoint scheme, and each figure is checked
against the frozen result (`results/model_results.json`).

`RUN_HEAVY = True` refits everything (about two minutes). `False` shows the frozen values and
reruns only the cheap baselines.

In [1]:
RUN_HEAVY = True

from common import *
import lightgbm, sklearn
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

print(f"scikit-learn {sklearn.__version__}, lightgbm {lightgbm.__version__} (frozen with 1.9.0 and 4.7.0)")
PAIRS = load_pairs()
BASE = PAIRS[PAIRS.contrast_type == "base_mut"].reset_index(drop=True)
CONTRACT = load(DATA / "redmi9t_feature_contract.json")
FEATURES = CONTRACT["feature_columns"]
ALIAS, COUNT = "d_usesLayoutDependentBuilder", "d_treeNonConstWidgetCount"
FROZEN = load(RES / "model_results.json")
EDA = load(DATA / "redmi9t_eda_summary.json")
ORACLE = load(RES / "posthoc_oracle.json")
FOLDS = load(DATA / "redmi9t_base_mut_stratified_group_kfold.json")["folds"]
y, groups = BASE.label.to_numpy(), BASE.group.to_numpy()

scikit-learn 1.9.0, lightgbm 4.7.0 (frozen with 1.9.0 and 4.7.0)


## The four learners

Five outer folds are read from the stored split file, so whole seed groups are held out. Three inner
group-disjoint folds choose the hyper-parameters by average precision, and the decision threshold is 0.5.

In [2]:
GRIDS = {
    "logreg_balanced": {"model__C": [0.03, 0.3, 3.0, 30.0]},
    "random_forest": {"model__max_depth": [4, 8, None], "model__min_samples_leaf": [2, 8]},
    "rbf_svm": {"model__C": [0.3, 1.0, 3.0, 10.0], "model__gamma": ["scale", 0.03]},
    "lightgbm": {"model__num_leaves": [3, 7, 15], "model__min_child_samples": [10, 25],
                 "model__learning_rate": [0.03, 0.10]},
}

def logistic(**kw):
    p = dict(class_weight="balanced", penalty="l2", solver="liblinear", max_iter=5000, random_state=SEED)
    p.update(kw)
    return Pipeline([("variance", VarianceThreshold()), ("scale", RobustScaler()), ("model", LogisticRegression(**p))])

def simple_logistic(**kw):  # no variance filter: for the one-column baselines
    p = dict(class_weight="balanced", penalty="l2", solver="liblinear", max_iter=5000, random_state=SEED)
    p.update(kw)
    return Pipeline([("scale", RobustScaler()), ("model", LogisticRegression(**p))])

ESTIMATORS = {
    "logreg_balanced": logistic(),
    "random_forest": Pipeline([("variance", VarianceThreshold()), ("model", RandomForestClassifier(
        n_estimators=300, class_weight="balanced_subsample", max_features="sqrt", random_state=SEED, n_jobs=1))]),
    "rbf_svm": Pipeline([("variance", VarianceThreshold()), ("scale", RobustScaler()),
                         ("model", SVC(probability=True, class_weight="balanced", random_state=SEED))]),
    "lightgbm": Pipeline([("variance", VarianceThreshold()), ("model", LGBMClassifier(
        n_estimators=300, class_weight="balanced", verbosity=-1, deterministic=True, random_state=SEED, n_jobs=1))]),
}

def nested_cv(frame, estimator, grid, columns):
    """Out-of-fold probabilities under the stored outer folds; inner folds are group-disjoint."""
    pos = pd.Series(range(len(frame)), index=frame["pair_id"])
    X, yy, g = frame[columns], frame["label"].to_numpy(), frame["group"].to_numpy()
    oof = np.full(len(frame), np.nan)
    for f in FOLDS:
        tr, te = pos.loc[f["train_pair_ids"]].to_numpy(), pos.loc[f["test_pair_ids"]].to_numpy()
        assert not set(g[tr]) & set(g[te]), "a seed group straddles train and test"
        if grid:
            inner = StratifiedGroupKFold(3, shuffle=True, random_state=SEED + 100 + f["fold"])
            model = GridSearchCV(clone(estimator), grid, scoring="average_precision", cv=inner, refit=True,
                                 error_score="raise", n_jobs=1).fit(X.iloc[tr], yy[tr], groups=g[tr]).best_estimator_
        else:
            model = clone(estimator).fit(X.iloc[tr], yy[tr])
        oof[te] = model.predict_proba(X.iloc[te])[:, 1]
    assert not np.isnan(oof).any()
    return oof

def group_bootstrap(stat, p, draws=2000):
    """Percentile 95% interval of a statistic, resampling whole seed groups."""
    uniq = np.unique(groups); rows = {u: np.flatnonzero(groups == u) for u in uniq}
    rng = np.random.default_rng(SEED); vals = []
    for _ in range(draws):
        idx = np.concatenate([rows[u] for u in rng.choice(uniq, size=len(uniq), replace=True)])
        if len(np.unique(y[idx])) == 2:
            vals.append(stat(y[idx], p[idx]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

OOF, AUC = {}, {}
for name in ESTIMATORS:
    if RUN_HEAVY:
        OOF[name] = nested_cv(BASE, ESTIMATORS[name], GRIDS[name], FEATURES)
        AUC[name] = roc_auc_score(y, OOF[name])
        check(f"AUC, {name}", AUC[name], FROZEN["model_comparison"][name]["roc_auc"], 1e-4)
        lo, hi = group_bootstrap(roc_auc_score, OOF[name])
        check(f"AUC CI low, {name}", lo, FROZEN["intervals"][name]["roc_auc"]["ci_low"], 1e-4)
        check(f"AUC CI high, {name}", hi, FROZEN["intervals"][name]["roc_auc"]["ci_high"], 1e-4)
    else:
        AUC[name] = FROZEN["model_comparison"][name]["roc_auc"]

table = pd.DataFrame({n: {"AUC": AUC[n], "AUC low": FROZEN["intervals"][n]["roc_auc"]["ci_low"],
                          "AUC high": FROZEN["intervals"][n]["roc_auc"]["ci_high"],
                          "MCC": FROZEN["model_comparison"][n]["mcc"],
                          "MCC low": FROZEN["intervals"][n]["mcc"]["ci_low"],
                          "MCC high": FROZEN["intervals"][n]["mcc"]["ci_high"]} for n in ESTIMATORS}).T
print("Table: the four learners on base_mut")
print(table.round(3).sort_values("AUC").to_string())
check("range, low", min(AUC.values()), 0.777, 5e-4)
check("range, high", max(AUC.values()), 0.828, 5e-4)
check("MCC range, low", table.MCC.min(), 0.432, 5e-4)
check("MCC range, high", table.MCC.max(), 0.489, 5e-4)

Table: the four learners on base_mut
                   AUC  AUC low  AUC high    MCC  MCC low  MCC high
logreg_balanced  0.777    0.728     0.828  0.464    0.365     0.561
rbf_svm          0.795    0.743     0.846  0.432    0.346     0.522
lightgbm         0.795    0.736     0.853  0.482    0.389     0.583
random_forest    0.828    0.778     0.876  0.489    0.376     0.602


## Against chance

The permutation test (200 shuffles, each re-running the whole nested procedure including tuning) takes too
long to repeat, so the frozen result is shown.

In [3]:
pt = FROZEN["permutation_test"]
check("permutation null mean", pt["null_mean"], 0.501, 5e-4)
check("permutation null sd", pt["null_std"], 0.044, 5e-4)
check("permutations", pt["n_perm"], 200)
check("permutation p (a bound: none of 200 reached 0.777)", pt["p_value"], 0.005, 5e-4)

## Baselines, the directive reference and the ceiling

Each baseline is one alternative explanation for a good score. The *directive oracle* knows only which
of the eight edits was applied (the identity of the original and the variant), through one-hot columns.

In [4]:
DIRECTIVES = [f"mutation_{i}" for i in range(1, 9)]
for side in ("left", "right"):
    for r in ["base", *DIRECTIVES]:
        BASE[f"{side}_{r}"] = (BASE[side] == r).astype(float)
DIRECTIVE_COLS = [f"{s}_{r}" for s in ("left", "right") for r in ["base", *DIRECTIVES]]

def cv_auc(columns, estimator, grid=None):
    return roc_auc_score(y, nested_cv(BASE, estimator, grid, columns))

mc = FROZEN["model_comparison"]
base_auc = {
    "prevalence": cv_auc(FEATURES[:1], DummyClassifier(strategy="prior")),
    "single feature: non-constant widget count": cv_auc([COUNT], simple_logistic(C=1.0)),
    "single feature: layout-dependent builder": cv_auc([ALIAS], simple_logistic(C=1.0)),
    "directive oracle": cv_auc(DIRECTIVE_COLS, logistic(C=1.0)),
}
for (label, got), key in zip(base_auc.items(), ["prevalence", "single_feature_count", "single_feature_alias", "directive_oracle"]):
    check(f"baseline AUC, {label}", got, mc[key]["roc_auc"], 1e-4)
    check(f"baseline MCC, {label}", mc[key]["mcc"], {"prevalence": 0.0, "single_feature_count": 0.108,
          "single_feature_alias": 0.393, "directive_oracle": 0.582}[key], 5e-4)
oi = FROZEN["intervals"]["directive_oracle"]["roc_auc"]
check("oracle interval, low", oi["ci_low"], 0.808, 5e-4)
check("oracle interval, high", oi["ci_high"], 0.882, 5e-4)

# Post-hoc: learner minus oracle, both scored on the same resampled contrasts (frozen, paired bootstrap).
for name, want in {"logreg_balanced": -0.066, "rbf_svm": -0.048, "lightgbm": -0.048, "random_forest": -0.015}.items():
    check(f"learner minus oracle, {name}", ORACLE["auc_minus_oracle"][name]["point"], want, 5e-4)
rf = ORACLE["auc_minus_oracle"]["random_forest"]["ci95"]
check("random forest minus oracle, CI low", rf[0], -0.044, 5e-4); check("random forest minus oracle, CI high", rf[1], 0.014, 5e-4)
check("logreg minus oracle, CI low", ORACLE["auc_minus_oracle"]["logreg_balanced"]["ci95"][0], -0.102, 5e-4)
check("logreg minus oracle, CI high", ORACLE["auc_minus_oracle"]["logreg_balanced"]["ci95"][1], -0.029, 5e-4)

# Accuracy needs its own denominator.
ceil = EDA["feature_resolution"]["attainable_ceiling"]["base_mut"]
m = FROZEN["logreg_balanced_detail"]["metrics"]
check("accuracy, balanced logistic regression", m["accuracy"], 0.741, 5e-4)
check("majority-class accuracy", ceil["majority_baseline_accuracy"], 0.581, 5e-4)
check("attainable ceiling", ceil["attainable_accuracy_ceiling"], 0.949, 5e-4)
check("distinct feature vectors", ceil["distinct_feature_vectors"], 225)
check("vectors with conflicting labels", ceil["vectors_with_conflicting_labels"], 13)

## Sensitivity of the range

Frozen sensitivity runs (`model_results.json`); the drift-adjusted range was shown in notebook 1.

In [5]:
s = FROZEN["sensitivity"]
check("AUC without the 42 ambiguous contrasts", s["ambiguity"]["unambiguous_only"]["roc_auc"], 0.809, 5e-4)
check("AUC with labels at delta > 0.474", s["endpoint"]["metrics"]["roc_auc"], 0.767, 5e-4)
check("mut_mut, logistic regression, 12 features", FROZEN["by_stratum"]["mut_mut"]["auc"]["point"], 0.843, 5e-4)
check("  its CI low", FROZEN["by_stratum"]["mut_mut"]["auc"]["ci_low"], 0.801, 5e-4)
check("  its CI high", FROZEN["by_stratum"]["mut_mut"]["auc"]["ci_high"], 0.881, 5e-4)
check("pooled strata, logistic regression", FROZEN["by_stratum"]["pooled"]["auc"]["point"], 0.828, 5e-4)
check("  its CI low", FROZEN["by_stratum"]["pooled"]["auc"]["ci_low"], 0.788, 5e-4)
check("  its CI high", FROZEN["by_stratum"]["pooled"]["auc"]["ci_high"], 0.867, 5e-4)

## What the models rely on

Permutation importance (mean drop in AUC, balanced logistic regression) is frozen. The count of
contrasts in which no feature moves is recomputed.

In [6]:
pi = FROZEN["interpretation"]["permutation_importance"]
imp = pd.DataFrame(pi).T.sort_values("mean", ascending=False)
print(imp.round(3).to_string())
check("usesLayoutDependentBuilder importance", pi["usesLayoutDependentBuilder"]["mean"], 0.149, 5e-4)
check("treeNonConstWidgetCount importance", pi["treeNonConstWidgetCount"]["mean"], 0.138, 5e-4)
check("helperWidgetCount importance", pi["helperWidgetCount"]["mean"], 0.115, 5e-4)
check("largest of the other nine", imp["mean"].iloc[3], 0.017, 5e-4)
check("rootBuildReturnsConstWidget coefficient", FROZEN["interpretation"]["coefficient_stability"]["rootBuildReturnsConstWidget"]["median"], -1.456, 5e-4)
check("rootBuildReturnsConstWidget importance", pi["rootBuildReturnsConstWidget"]["mean"], 0.001, 5e-4)

zero = (BASE[FEATURES] == 0).all(axis=1)
check("contrasts with every delta zero", zero.sum(), 20)
check("  share of base_mut", zero.mean(), 0.060, 5e-4)
check("  share of them slower", BASE.label[zero].mean(), 0.05, 5e-4)

                              mean    std
usesLayoutDependentBuilder   0.149  0.038
treeNonConstWidgetCount      0.138  0.047
helperWidgetCount            0.115  0.041
treeMaxWidgetNestingDepth    0.017  0.025
treeCyclomaticComplexity     0.016  0.014
treeIterationCount           0.014  0.014
treeListRenderingStrategy    0.010  0.009
treeConstWidgetCount         0.002  0.005
helperReferenceCount         0.001  0.007
rootBuildReturnsConstWidget  0.001  0.002
valueObjectAllocCount       -0.001  0.009
iterationWidgetCount        -0.010  0.015


## Ablations

Removing `usesLayoutDependentBuilder` costs the linear learner 0.099 AUC. The other three learners are
refitted on the eleven-feature design (all four are checked against the frozen run).

In [7]:
ab = FROZEN["ablations"]
print(pd.DataFrame(ab).T[["roc_auc", "mcc"]].round(3).to_string())
for key, auc, mcc in [("full_12", 0.777, 0.464), ("no_alias", 0.678, 0.162), ("counts_only", 0.677, 0.162),
                      ("top_3_in_fold", 0.713, 0.406), ("top_6_in_fold", 0.736, 0.375)]:
    check(f"ablation AUC, {key}", ab[key]["roc_auc"], auc, 5e-4)
    check(f"ablation MCC, {key}", ab[key]["mcc"], mcc, 5e-4)
check("cost of removing the alias feature", FROZEN["directive_confound"]["alias_ablation_auc_cost"], 0.099, 5e-4)

NO_ALIAS = [f for f in FEATURES if f != ALIAS]
no_alias_auc = {}
if RUN_HEAVY:
    for name in ESTIMATORS:
        no_alias_auc[name] = cv_auc(NO_ALIAS, ESTIMATORS[name], GRIDS[name])
    check("no-alias AUC, logreg (frozen ablation)", no_alias_auc["logreg_balanced"], ab["no_alias"]["roc_auc"], 1e-4)
    check("no-alias AUC, random forest", no_alias_auc["random_forest"], 0.828, 5e-4)
    check("no-alias AUC, lightgbm", no_alias_auc["lightgbm"], 0.807, 5e-4)
    vals = pd.Series(no_alias_auc)
    check("no-alias range, low", vals.min(), 0.678, 5e-4)
    check("spread of the four learners, 12 features", max(AUC.values()) - min(AUC.values()), 0.051, 5e-4)
    print(vals.round(4).to_string())
    print("spread, no alias:", round(vals.max() - vals.min(), 4))
    check("spread of the four learners without the alias feature", vals.max() - vals.min(), 0.150, 5e-4)

                roc_auc       mcc
full_12        0.777128  0.464108
no_alias       0.678086  0.161883
counts_only    0.676818  0.161883
top_3_in_fold  0.713125  0.406119
top_6_in_fold  0.735733  0.375461


logreg_balanced    0.6781
random_forest      0.8281
rbf_svm            0.7211
lightgbm           0.8073
spread, no alias: 0.15


## RQ2: screening utility

All four learners at the default threshold 0.5, which is not tuned. Flag counts, precision and recall are
recomputed from the out-of-fold scores; the precision intervals over seed groups and the directive reference come
from the frozen post-hoc run (`results/posthoc_oracle.json`).

In [8]:
AT = ORACLE["at_threshold_0.5"]; PCI = ORACLE["precision_95ci_at_0.5"]
want = {"logreg_balanced": (131, 0.702, 0.662, 0.617, 0.794), "random_forest": (131, 0.718, 0.676, 0.636, 0.805),
        "rbf_svm": (130, 0.685, 0.640, 0.607, 0.770), "lightgbm": (143, 0.692, 0.712, 0.611, 0.784)}
rows = {}
for name, (flags, prec, rec, lo, hi) in want.items():
    if RUN_HEAVY:
        pred = OOF[name] >= 0.5
        tp = int((pred & (y == 1)).sum())
        check(f"{name}: flagged", int(pred.sum()), flags)
        check(f"{name}: precision", tp / pred.sum(), prec, 5e-4)
        check(f"{name}: recall", tp / (y == 1).sum(), rec, 5e-4)
    check(f"{name}: flagged (frozen)", AT[name]["flagged"], flags)
    check(f"{name}: precision (frozen)", AT[name]["precision"], prec, 5e-4)
    check(f"{name}: recall (frozen)", AT[name]["recall"], rec, 5e-4)
    check(f"{name}: precision CI low", PCI[name][0], lo, 5e-4); check(f"{name}: precision CI high", PCI[name][1], hi, 5e-4)
    rows[name] = (AT[name]["flagged"], AT[name]["precision"], PCI[name][0], PCI[name][1], AT[name]["recall"])
rows["directive_oracle"] = (AT["directive_oracle"]["flagged"], AT["directive_oracle"]["precision"],
                            PCI["directive_oracle"][0], PCI["directive_oracle"][1], AT["directive_oracle"]["recall"])
print(pd.DataFrame(rows, index=["flagged", "precision", "CI low", "CI high", "recall"]).T.round(3).to_string())
check("prevalence", y.mean(), 0.419, 5e-4)
check("precision range, low", min(AT[n]["precision"] for n in want), 0.685, 5e-4)
check("precision range, high", max(AT[n]["precision"] for n in want), 0.718, 5e-4)
check("oracle flags at 0.5", AT["directive_oracle"]["flagged"], 128)
check("oracle precision at 0.5", AT["directive_oracle"]["precision"], 0.781, 5e-4)
check("oracle precision CI low", PCI["directive_oracle"][0], 0.711, 5e-4)
check("oracle precision CI high", PCI["directive_oracle"][1], 0.853, 5e-4)
check("oracle expected precision at 131 flags", ORACLE["oracle_at_131_flags"]["expected_precision"], 0.768, 5e-4)

                  flagged  precision  CI low  CI high  recall
logreg_balanced     131.0      0.702   0.617    0.794   0.662
random_forest       131.0      0.718   0.636    0.805   0.676
rbf_svm             130.0      0.685   0.607    0.770   0.640
lightgbm            143.0      0.692   0.611    0.784   0.712
directive_oracle    128.0      0.781   0.711    0.853   0.719


## Figures

The figures for this notebook's sections, written to `figures/` as PNG (300 dpi) and SVG. They plot only values checked above. The images carry no title or note; the caption in the report explains them.

In [9]:
plt = figure_style()
iv = FROZEN["intervals"]

# AUC of the four learners and the reference points, with their 95% group-bootstrap intervals.
rows = [("random forest", "random_forest", AUC["random_forest"], True),
        ("LightGBM", "lightgbm", AUC["lightgbm"], True),
        ("RBF support vector machine", "rbf_svm", AUC["rbf_svm"], True),
        ("balanced logistic regression", "logreg_balanced", AUC["logreg_balanced"], True),
        ("directive oracle (reference)", "directive_oracle", base_auc["directive oracle"], False),
        ("single feature: layout-dependent builder", "single_feature_alias",
         base_auc["single feature: layout-dependent builder"], False),
        ("single feature: non-constant widget count", "single_feature_count",
         base_auc["single feature: non-constant widget count"], False),
        ("prevalence", "prevalence", base_auc["prevalence"], False)]
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.axvspan(min(AUC.values()), max(AUC.values()), color=LIGHT, alpha=0.5, lw=0)
for i, (label, key, point, learner) in enumerate(rows):
    lo_, hi_ = iv[key]["roc_auc"]["ci_low"], iv[key]["roc_auc"]["ci_high"]
    ax.errorbar(point, i, xerr=[[point - lo_], [hi_ - point]], fmt="o" if learner else "s", ms=6,
                color=DARK if learner else MID, mfc=DARK if learner else "white", capsize=3, lw=1.2)
    ax.text(hi_ + 0.01, i, f"{point:.3f}", va="center", fontsize=7.5)
ax.axvline(0.5, color=MID, ls=":", lw=1)
ax.axhline(3.5, color=LIGHT, lw=0.8)
ax.set_yticks(range(len(rows)), [r[0] for r in rows])
ax.invert_yaxis()
ax.set_xlim(0.35, 1.0)
ax.set_xlabel("ROC AUC, base-to-variant contrasts (95% block bootstrap over seed groups)")
save_figure(fig, "learner_range")

# Permutation importance, balanced logistic regression.
order = imp.sort_values("mean")
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(range(len(order)), order["mean"], xerr=order["std"], color=MID, ecolor=DARK, capsize=2)
ax.set_yticks(range(len(order)), order.index, fontsize=7.5, family="DejaVu Sans Mono")
ax.axvline(0, color=DARK, lw=0.8)
ax.set_xlabel("mean drop in AUC when the feature is permuted (± sd)")
save_figure(fig, "feature_importance")


wrote figures/learner_range.png and .svg


wrote figures/feature_importance.png and .svg


## Post-hoc result figures (2026-10-07)

Three descriptive views of the out-of-fold scores computed above: where the disagreements with the label
fall along the measured δ, how many slower contrasts a reviewer would catch by checking the highest-scored
contrasts first, and the ROC curves behind the AUC range. They are post-hoc and descriptive: no learner,
threshold or setting is chosen from them. The values they show are written to
`results/posthoc_result_figures.json`. They need `RUN_HEAVY = True`.

In [10]:
if RUN_HEAVY:
    from sklearn.metrics import roc_curve
    NAMES = {"random_forest": "random forest", "lightgbm": "LightGBM",
             "rbf_svm": "RBF support vector machine", "logreg_balanced": "balanced logistic regression"}
    LINE = {"random_forest": "-", "lightgbm": "--", "rbf_svm": "-.", "logreg_balanced": ":"}
    ORACLE_OOF = nested_cv(BASE, logistic(C=1.0), None, DIRECTIVE_COLS)
    check("directive oracle AUC, recomputed scores", roc_auc_score(y, ORACLE_OOF), mc["directive_oracle"]["roc_auc"], 1e-4)
    SCORES = {**{n: OOF[n] for n in NAMES}, "directive_oracle": ORACLE_OOF}
    delta = BASE.cliffs_delta.to_numpy()
    ambiguous = BASE.ambiguous.astype(str).str.lower().eq("true").to_numpy()
    check("ambiguous contrasts", ambiguous.sum(), 42)
    POST = {"status": "post-hoc, descriptive, 2026-10-07: views of the frozen out-of-fold scores; nothing selected or tuned",
            "stratum": "base_mut", "n": int(len(y)), "slower": int(y.sum()), "threshold": 0.5}

    # 1. Agreement with the label at 0.5, by where the measured delta lies relative to the 0.33 cut.
    BINS = [("clearly faster", "below −0.147", delta < -0.147),
            ("no clear change", "−0.147 to 0.147", (delta >= -0.147) & (delta < 0.147)),
            ("slightly slower", "0.147 to 0.33", (delta >= 0.147) & (delta <= CUT)),
            ("slower", "0.33 to 0.474", (delta > CUT) & (delta < 0.474)),
            ("clearly slower", "0.474 and above", delta >= 0.474)]
    assert sum(m.sum() for _, _, m in BINS) == len(y)
    POST["agreement_by_delta_bin"] = {}
    for name, label, mask in BINS:
        row = {"range": label, "n": int(mask.sum()), "slower": int(y[mask].sum())}
        for key, s in SCORES.items():
            k = int(((s[mask] >= 0.5).astype(int) == y[mask]).sum())
            lo_, hi_ = clopper_pearson(k, mask.sum())
            row[key] = {"agree": k, "share": round(k / mask.sum(), 4), "ci95": [round(lo_, 4), round(hi_, 4)]}
        POST["agreement_by_delta_bin"][name] = row
    agree = pd.DataFrame({b: {k: v["share"] for k, v in r.items() if isinstance(v, dict)} | {"n": r["n"]}
                          for b, r in POST["agreement_by_delta_bin"].items()})
    print("Agreement with the label at 0.5, by measured delta"); print(agree.round(3).to_string())
    near = (delta >= 0.147) & (delta < 0.474)
    POST["agreement_near_vs_far_from_cut"] = {
        key: {"near_0.147_to_0.474": round(float(((s[near] >= 0.5) == y[near]).mean()), 4),
              "elsewhere": round(float(((s[~near] >= 0.5) == y[~near]).mean()), 4),
              "n_near": int(near.sum()), "n_elsewhere": int((~near).sum())} for key, s in SCORES.items()}
    print(pd.DataFrame(POST["agreement_near_vs_far_from_cut"]).T.to_string())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.9), gridspec_kw={"width_ratios": [1.15, 1]})
    s = OOF["logreg_balanced"]; right = (s >= 0.5).astype(int) == y
    for amb, marker in ((False, "o"), (True, "^")):
        m = ambiguous == amb
        ax1.scatter(delta[m & right], s[m & right], s=14, marker=marker, facecolors="white", edgecolors=MID, lw=0.7)
        ax1.scatter(delta[m & ~right], s[m & ~right], s=16, marker=marker, color=DARK, lw=0)
    ax1.axvline(CUT, color=DARK, ls=":", lw=1); ax1.axhline(0.5, color=DARK, ls=":", lw=1)
    for x in (-0.147, 0.147, 0.474):
        ax1.axvline(x, color=LIGHT, lw=0.8, zorder=0)
    ax1.text(CUT + 0.02, 0.02, "label cut δ = 0.33", fontsize=7.5, color=DARK)
    ax1.set_xlim(-1.02, 1.02); ax1.set_ylim(0, 1)
    ax1.set_xlabel("measured Cliff's δ (variant against base; > 0 means slower)")
    ax1.set_ylabel("predicted probability of slower")
    ax1.scatter([], [], s=14, facecolors="white", edgecolors=MID, label="agrees with the label")
    ax1.scatter([], [], s=16, color=DARK, label="disagrees")
    ax1.scatter([], [], s=14, marker="^", facecolors="white", edgecolors=MID, label="ambiguous contrast")
    ax1.legend(loc="upper left", fontsize=7, frameon=False)
    ax1.text(-0.12, 1.02, "(a)", transform=ax1.transAxes, fontsize=9, fontweight="bold")

    offsets = {"random_forest": -0.24, "lightgbm": -0.12, "rbf_svm": 0.0, "logreg_balanced": 0.12, "directive_oracle": 0.24}
    marks = {"random_forest": "o", "lightgbm": "s", "rbf_svm": "D", "logreg_balanced": "v", "directive_oracle": "s"}
    for key, dx in offsets.items():
        oracle = key == "directive_oracle"
        for i, (name, _, _) in enumerate(BINS):
            r = POST["agreement_by_delta_bin"][name][key]
            ax2.errorbar(i + dx, r["share"], yerr=[[r["share"] - r["ci95"][0]], [r["ci95"][1] - r["share"]]],
                         fmt=marks[key], ms=4.5, color=MID if oracle else DARK, mfc="white" if oracle else DARK,
                         capsize=2, lw=0.9)
        ax2.errorbar([], [], fmt=marks[key], ms=4.5, color=MID if oracle else DARK, mfc="white" if oracle else DARK,
                     label="directive oracle (reference)" if oracle else NAMES[key])
    ax2.axvspan(1.5, 3.5, color=LIGHT, alpha=0.35, lw=0, zorder=0)
    ax2.set_xticks(range(len(BINS)), [f"{n}\n{l}\nn = {POST['agreement_by_delta_bin'][n]['n']}" for n, l, _ in BINS], fontsize=6.8)
    ax2.set_xlabel("measured Cliff's δ", fontsize=8)
    ax2.set_ylim(0, 1.05); ax2.set_ylabel("share agreeing with the label at 0.5 (95% CI)")
    ax2.legend(loc="lower left", fontsize=6.5, frameon=False)
    ax2.text(-0.12, 1.02, "(b)", transform=ax2.transAxes, fontsize=9, fontweight="bold")
    fig.tight_layout()
    save_figure(fig, "errors_by_effect")

    # 2. Screening gain: review contrasts from the highest score down; tied scores count in expectation.
    def gain(score):
        uniq = np.unique(score)[::-1]
        n_seen = np.concatenate([[0], np.cumsum([(score == u).sum() for u in uniq])])
        caught = np.concatenate([[0], np.cumsum([y[score == u].sum() for u in uniq])])
        return n_seen / len(y), caught / y.sum()
    GAIN = {k: gain(sc) for k, sc in SCORES.items()}
    POST["recall_when_reviewing_top_share"] = {
        f"{int(q * 100)}%": {k: round(float(np.interp(q, *GAIN[k])), 4) for k in SCORES} | {"random": q}
        for q in (0.2, 0.3, 0.4, 0.5)}
    POST["operating_point_at_0.5"] = {k: {"flagged": int((SCORES[k] >= 0.5).sum()),
                                          "reviewed_share": round(float((SCORES[k] >= 0.5).mean()), 4),
                                          "recall": round(float(((SCORES[k] >= 0.5) & (y == 1)).sum() / y.sum()), 4)}
                                      for k in SCORES}
    for k in NAMES:
        check(f"gain-curve operating point, {k}", POST["operating_point_at_0.5"][k]["flagged"], AT[k]["flagged"])
        check(f"gain-curve operating recall, {k}", POST["operating_point_at_0.5"][k]["recall"], AT[k]["recall"], 5e-4)
    print("Share of the slower contrasts caught when reviewing the top share"); print(pd.DataFrame(POST["recall_when_reviewing_top_share"]).round(3).to_string())

    prev = y.mean()
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot([0, prev, 1], [0, 1, 1], color=LIGHT, lw=1.2, label="perfect ranking")
    ax.plot([0, 1], [0, 1], color=MID, ls=":", lw=1, label="random review")
    ax.plot(*GAIN["directive_oracle"], color=MID, ls="--", lw=1.4, label="directive oracle (reference)")
    for k in NAMES:
        ax.plot(*GAIN[k], color=DARK, ls=LINE[k], lw=1.3, label=NAMES[k])
        op = POST["operating_point_at_0.5"][k]
        ax.plot(op["reviewed_share"], op["recall"], "o", ms=4, color=DARK)
    ax.annotate("threshold 0.5\n(flagged by each learner)", xy=(POST["operating_point_at_0.5"]["logreg_balanced"]["reviewed_share"],
                POST["operating_point_at_0.5"]["logreg_balanced"]["recall"]), xytext=(0.5, 0.42),
                fontsize=7.5, arrowprops=dict(arrowstyle="-", color=MID, lw=0.8))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.set_xlabel("share of contrasts reviewed, highest predicted risk first")
    ax.set_ylabel(f"share of the {int(y.sum())} slower contrasts caught")
    ax.legend(loc="lower right", fontsize=7, frameon=False)
    save_figure(fig, "screening_gain")

    # 3. ROC curves of the four learners, with the directive oracle as reference.
    POST["roc_auc"] = {k: round(float(roc_auc_score(y, sc)), 4) for k, sc in SCORES.items()}
    for k in NAMES:
        check(f"ROC figure AUC, {k}", POST["roc_auc"][k], FROZEN["model_comparison"][k]["roc_auc"], 1e-4)
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.plot([0, 1], [0, 1], color=MID, ls=":", lw=1, label="chance (0.500)")
    fpr, tpr, _ = roc_curve(y, ORACLE_OOF)
    ax.plot(fpr, tpr, color=MID, ls="--", lw=1.4, label=f"directive oracle, reference ({POST['roc_auc']['directive_oracle']:.3f})")
    for k in NAMES:
        fpr, tpr, _ = roc_curve(y, OOF[k])
        ax.plot(fpr, tpr, color=DARK, ls=LINE[k], lw=1.3, label=f"{NAMES[k]} ({AUC[k]:.3f})")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.01); ax.set_aspect("equal")
    ax.set_xlabel("false-positive rate (not-slower contrasts flagged)")
    ax.set_ylabel("true-positive rate (slower contrasts flagged)")
    ax.legend(loc="lower right", fontsize=7, frameon=False, title="ROC AUC", title_fontsize=7)
    save_figure(fig, "roc_four_learners")

    (RES / "posthoc_result_figures.json").write_text(json.dumps(POST, indent=2, ensure_ascii=False) + "\n")
    print("wrote results/posthoc_result_figures.json")


Agreement with the label at 0.5, by measured delta
                  clearly faster  no clear change  slightly slower  slower  clearly slower
random_forest              0.853            0.818            0.729    0.35           0.731
lightgbm                   0.838            0.766            0.688    0.45           0.756
rbf_svm                    0.882            0.766            0.688    0.45           0.672
logreg_balanced            0.853            0.831            0.667    0.55           0.681
directive_oracle           0.941            0.818            0.792    0.45           0.765
n                         68.000           77.000           48.000   20.00         119.000
                  near_0.147_to_0.474  elsewhere  n_near  n_elsewhere
random_forest                  0.6176     0.7879    68.0        264.0
lightgbm                       0.6176     0.7803    68.0        264.0
rbf_svm                        0.6176     0.7538    68.0        264.0
logreg_balanced                0

wrote figures/errors_by_effect.png and .svg
Share of the slower contrasts caught when reviewing the top share
                    20%    30%    40%    50%
random_forest     0.406  0.554  0.682  0.777
lightgbm          0.399  0.537  0.684  0.777
rbf_svm           0.378  0.551  0.646  0.755
logreg_balanced   0.381  0.530  0.675  0.748
directive_oracle  0.422  0.603  0.727  0.794
random            0.200  0.300  0.400  0.500


wrote figures/screening_gain.png and .svg


wrote figures/roc_four_learners.png and .svg
wrote results/posthoc_result_figures.json


## Result

Every figure above is reproduced.

In [11]:
report()

131 / 131 reported figures reproduced


,claim,value,reported,ok
0,"AUC, logreg_balanced",0.777128,0.777128,True
1,"AUC CI low, logreg_balanced",0.728121,0.728121,True
2,"AUC CI high, logreg_balanced",0.828296,0.828296,True
3,"AUC, random_forest",0.827785,0.827785,True
4,"AUC CI low, random_forest",0.778311,0.778311,True
...,...,...,...,...
126,"gain-curve operating recall, logreg_balanced",0.661900,0.661871,True
127,"ROC figure AUC, random_forest",0.827800,0.827785,True
128,"ROC figure AUC, lightgbm",0.795200,0.795150,True
129,"ROC figure AUC, rbf_svm",0.795100,0.795132,True
